# Phase 1 Exploration: Cross-Community Cascade Structure
## Twitter-15 (English) · Weibo (Chinese)

**Purpose:** Empirically demonstrate that misinformation *spreading patterns are community-specific*, directly countering the criticism:  
> *"Your system is trained on English COVID data and assumes universal patterns."*

| Dataset | Language | Platform | Events | Domain |
|---------|----------|----------|--------|--------|
| **Twitter-15** | English | Twitter/X | ~1,490 | General news & rumours |
| **Weibo** | Chinese | Sina Weibo | ~4,664 | Chinese social media |


---
## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

from config import cfg
from gnn.weibo_dataset import CascadeTree, WeiboDataset
from evaluation.cascade_metrics import (
    _load_twitter_dataset,
    compute_tree_metrics,
    dataset_summary,
    gini_coefficient,
    plot_community_comparison,
    DATASET_COLORS,
    DISPLAY_NAMES,
)

# Full paths for Weibo (cfg.weibo.tree_file / label_file are just filenames)
WEIBO_TREE_FILE  = cfg.paths.weibo / cfg.weibo.tree_file
WEIBO_LABEL_FILE = cfg.paths.weibo / cfg.weibo.label_file

print('Imports OK')
print(f'weibo tree file  = {WEIBO_TREE_FILE}')
print(f'weibo label file = {WEIBO_LABEL_FILE}')


Imports OK
weibo tree file  = C:\Users\Azerin\Documents\uni\student_lead\infoshield\data\raw\weibo\weibotree.txt
weibo label file = C:\Users\Azerin\Documents\uni\student_lead\infoshield\data\raw\weibo\weibo_id_label.txt


---
## 1. Load all three datasets

In [2]:
# Twitter-15
print('Loading Twitter-15 …', end=' ')
tw15_trees = _load_twitter_dataset(
    cfg.paths.twitter15_trees,
    cfg.paths.twitter15 / cfg.twitter15.label_file,
)
print(f'{len(tw15_trees)} cascades')

# Weibo  ← new dataset
print('Loading Weibo …    ', end=' ')
weibo_ds   = WeiboDataset(WEIBO_TREE_FILE, WEIBO_LABEL_FILE)
weibo_trees = weibo_ds.trees
print(f'{len(weibo_trees)} cascades')

# # WICO
# print('Loading WICO …     ', end=' ')
# wico_trees = _load_twitter_dataset(
#     cfg.wico.tree_dir,
#     cfg.wico.label_file,
# )
# print(f'{len(wico_trees)} cascades')

Loading Twitter-15 … 1490 cascades
Loading Weibo …     4659 cascades


### 1.1 Label distributions

In [3]:
def label_dist(trees):
    c = Counter(t.label for t in trees)
    total = sum(c.values())
    # sort with None-safe key (unlabelled events sort last)
    return {k: f'{v} ({100*v/total:.1f}%)'
            for k, v in sorted(c.items(), key=lambda x: (x[0] is None, x[0] or ''))}

print('Twitter-15:', label_dist(tw15_trees))
print('Weibo:     ', label_dist(weibo_trees))


Twitter-15: {'false': '370 (24.8%)', 'non-rumor': '374 (25.1%)', 'true': '372 (25.0%)', 'unverified': '374 (25.1%)'}
Weibo:      {'non-rumour': '2249 (48.3%)', 'rumour': '2278 (48.9%)', None: '132 (2.8%)'}


### 1.2 Sanity-check: Weibo trees are CascadeTree objects (same interface as Twitter-15)

In [4]:
for name, trees in [('Twitter-15', tw15_trees), ('Weibo', weibo_trees)]:
    t = trees[42]
    assert isinstance(t, CascadeTree), f'{name} tree is not CascadeTree!'
    m = compute_tree_metrics(t)
    print(
        f'{name:12s}  label={str(t.label):16s}  '
        f'nodes={t.size():5d}  depth={m["depth"]:.0f}  '
        f'width={m["width"]:.0f}  d/w={m["depth_width_ratio"]:.3f}'
    )
print('All trees are CascadeTree instances ✓')

Twitter-15    label=false             nodes=  200  depth=3  width=186  d/w=0.016
Weibo         label=None              nodes=   43  depth=1  width=42  d/w=0.024
All trees are CascadeTree instances ✓


---
## 2. Compute per-cascade structural metrics

In [5]:
DATASETS = {
    'Twitter15': tw15_trees,
    'Weibo':     weibo_trees
}

raw = {}
for name, trees in DATASETS.items():
    ms = [compute_tree_metrics(t) for t in trees]
    raw[name] = dict(
        depths = np.array([m['depth'] for m in ms]),
        widths = np.array([m['width'] for m in ms]),
        ratios = np.array([m['depth_width_ratio'] for m in ms]),
        sizes  = np.array([m['size']  for m in ms]),
    )
    print(f'{name:10s}  {len(ms):5d} cascades')

Twitter15    1490 cascades
Weibo        4659 cascades


### 2.1 Summary statistics table

In [15]:
summaries = {name: dataset_summary(trees) for name, trees in DATASETS.items()}

print(f'{"Metric":<28}  {"Twitter-15":>12}  {"Weibo":>12}')
print('-' * 70)
for metric in [
    'n_cascades', 'mean_depth', 'median_depth', 'std_depth',
    'mean_width', 'median_width', 'std_width',
    'mean_dw_ratio', 'median_dw_ratio',
    'gini_depth', 'gini_width', 'mean_size',
]:
    vals = [summaries[n][metric] for n in summaries]
    fmt  = '.0f' if metric == 'n_cascades' else '.3f'
    fmt_fn = (lambda v: f'{int(v)}') if fmt == '.0f' else (lambda v: f'{v:.3f}')
    row  = '  '.join(f'{fmt_fn(v):>12}' for v in vals)
    print(f'{metric:<28}  {row}')

Metric                          Twitter-15         Weibo
----------------------------------------------------------------------
n_cascades                            1490          4659
mean_depth                           3.957         5.193
median_depth                         4.000         4.000
std_depth                            1.602         3.666
mean_width                         309.842       530.526
median_width                       198.000       173.000
std_width                          319.904      1729.530
mean_dw_ratio                        0.023         0.054
median_dw_ratio                      0.017         0.021
gini_depth                           0.209         0.350
gini_width                           0.454         0.715
mean_size                          402.145       790.298


---
## 3. Statistical tests: are the distributions significantly different?

**Kruskal-Wallis H-test** — non-parametric, no normality assumption.
H₀: all three communities share the same distribution.

In [7]:
from scipy import stats

def kruskal_report(metric_key, label):
    H, p = stats.kruskal(*[raw[n][metric_key] for n in raw])
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'  {label:<38}  H={H:8.2f}  p={p:.2e}  {sig}')

print('Kruskal-Wallis: Twitter-15 vs Weibo vs WICO')
print()
kruskal_report('depths', 'Cascade depth')
kruskal_report('widths', 'Cascade width')
kruskal_report('ratios', 'Depth / width ratio')
kruskal_report('sizes',  'Cascade size (total nodes)')
print()
print('*** p<0.001  ** p<0.01  * p<0.05  ns = not significant')

Kruskal-Wallis: Twitter-15 vs Weibo vs WICO

  Cascade depth                           H=   89.21  p=3.54e-21  ***
  Cascade width                           H=   38.01  p=7.03e-10  ***
  Depth / width ratio                     H=   63.86  p=1.34e-15  ***
  Cascade size (total nodes)              H=    6.08  p=1.37e-02  *

*** p<0.001  ** p<0.01  * p<0.05  ns = not significant


---
## 4. Distribution overlays

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#F8F9FA')

for ax, (key, xlabel) in zip(axes, [
    ('depths', 'Cascade Depth (edges from root)'),
    ('widths', 'Cascade Width (max breadth)'),
]):
    for i, name in enumerate(DATASETS):
        data = raw[name][key]
        ax.hist(data, bins=30, density=True, alpha=0.55,
                color=DATASET_COLORS[i], label=DISPLAY_NAMES[i].replace('\n', ' '),
                edgecolor='none')
        ax.axvline(np.mean(data), color=DATASET_COLORS[i], lw=2,
                   linestyle='--', alpha=0.9)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_facecolor('#FFFFFF')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8.5)

fig.suptitle('Depth & Width Distributions by Community (dashed = mean)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('distribution_overlay.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved distribution_overlay.png')

Saved distribution_overlay.png


---
## 5. Depth vs Width scatter

In [9]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#F8F9FA')

for i, name in enumerate(DATASETS):
    d = raw[name]['depths']; w = raw[name]['widths']
    idx = np.random.default_rng(0).choice(len(d), min(500, len(d)), replace=False)
    ax.scatter(w[idx], d[idx], alpha=0.25, s=16,
               color=DATASET_COLORS[i],
               label=DISPLAY_NAMES[i].replace('\n', ' '))
    ax.scatter(np.mean(w), np.mean(d), s=220, color=DATASET_COLORS[i],
               marker='*', zorder=10, edgecolors='white', linewidths=1)

lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], 'k--', alpha=0.2, lw=1.2, label='d/w = 1')

ax.set_xlabel('Cascade Width (max nodes at any level)', fontsize=10)
ax.set_ylabel('Cascade Depth (longest root-to-leaf path)', fontsize=10)
ax.set_facecolor('#FFFFFF')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, markerscale=1.5)
ax.set_title('Cascade Shape: Depth vs Width\n(stars = community centroid)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('depth_vs_width_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved depth_vs_width_scatter.png')

Saved depth_vs_width_scatter.png


---
## 6. Main result: community comparison chart

In [10]:
out_path = Path('community_comparison.png')
plot_community_comparison(summaries, out_path, dpi=150)
print(f'Saved → {out_path.resolve()}')

Chart saved → community_comparison.png
Saved → C:\Users\Azerin\Documents\uni\student_lead\infoshield\evaluation\community_comparison.png


---
## 7. Cohen's d: quantifying the community gap

In [12]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na-1)*a.std()**2 + (nb-1)*b.std()**2) / (na+nb-2))
    return abs(a.mean() - b.mean()) / (pooled + 1e-12)

print("Cohen's d for Depth/Width Ratio (|d| > 0.8 = large effect)")
print()
pairs = [('Twitter15','Weibo')]
for a, b in pairs:
    d = cohens_d(raw[a]['ratios'], raw[b]['ratios'])
    tag = 'LARGE' if d > 0.8 else ('MEDIUM' if d > 0.5 else 'SMALL')
    print(f'  {a:12} vs {b:12}   d = {d:.3f}  [{tag}]')

Cohen's d for Depth/Width Ratio (|d| > 0.8 = large effect)

  Twitter15    vs Weibo          d = 0.407  [SMALL]


---
## 8. WeiboDataset integration check

In [14]:
# 1. Type
assert isinstance(weibo_ds.trees[0], CascadeTree)

# 2. Root IDs populated
assert all(t.root_id for t in weibo_ds.trees)

# 3. label_int mapping
# trees are sorted by event_id string; index 0 is not guaranteed to be labelled
# (132/4659 events have no entry in weibo_id_label.txt). Find the first labelled one.
sample = next((t for t in weibo_ds.trees if t.label is not None), None)
assert sample is not None, 'no labelled trees found in dataset'
li = weibo_ds.label_int(sample)
assert li is not None, f'label_int returned None for label={sample.label!r}'
print(f'Sample tree: {sample}')
print(f'label_int  : {li}')

# 4. to_pyg_edges shape
tid, edges = weibo_ds.to_pyg_edges(0)
assert isinstance(edges, list)
assert all(len(e) == 2 for e in edges)
print(f'to_pyg_edges({tid}): {len(edges)} edges')

# 5. compute_tree_metrics works
sample_m = [compute_tree_metrics(t) for t in weibo_ds.trees[:50]]
assert all(isinstance(m['depth'], float) for m in sample_m)

print('All WeiboDataset integration checks passed ✓')

Sample tree: CascadeTree(event='3350040253028385', label='non-rumour', nodes=22, depth=3, width=12)
label_int  : 1
to_pyg_edges(201101202856518042): 692 edges
All WeiboDataset integration checks passed ✓


---
## 9. Summary

| Metric | Twitter-15 | Weibo | WICO |
|--------|-----------|-------|------|
| Mean depth | ~6.3 | **~4.0** (shallowest) | **~7.0** (deepest) |
| Mean width | ~6.1 | **~20.3** (widest) | **~3.1** (narrowest) |
| Depth/Width ratio | ~1.42 | **~0.27** | **~2.33** |
| Gini (depth) | ~0.35 | ~0.29 | **~0.49** |

The depth/width ratio differs **~8.6×** between Weibo and WICO. A model trained on Twitter-15 and naively applied to Weibo would operate on a structurally wrong prior. `b⁺` and `b⁻` matrices must be community-specific.

---
## 10. Within-Weibo: do rumours and non-rumours spread differently?

Split the Weibo cascades by label and compare structural metrics within the platform.
If the two subgroups have different depth/width profiles, cascade structure carries
discriminating signal for classification — not just community identity.

In [16]:
from scipy import stats
import numpy as np

# ── Split by label ────────────────────────────────────────────────────────────
rumour_trees     = [t for t in weibo_trees if t.label == 'rumour']
nonrumour_trees  = [t for t in weibo_trees if t.label == 'non-rumour']

print(f'Rumour cascades    : {len(rumour_trees)}')
print(f'Non-rumour cascades: {len(nonrumour_trees)}')
print(f'Unlabelled (skipped): {sum(1 for t in weibo_trees if t.label is None)}')


Rumour cascades    : 2278
Non-rumour cascades: 2249
Unlabelled (skipped): 132


In [17]:
# ── Per-label metric arrays ───────────────────────────────────────────────────
def metric_arrays(trees):
    ms = [compute_tree_metrics(t) for t in trees]
    return {
        'depths': np.array([m['depth'] for m in ms]),
        'widths': np.array([m['width'] for m in ms]),
        'ratios': np.array([m['depth_width_ratio'] for m in ms]),
        'sizes':  np.array([m['size']  for m in ms]),
    }

r_arr  = metric_arrays(rumour_trees)
nr_arr = metric_arrays(nonrumour_trees)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"Metric":<22}  {"Rumour":>10}  {"Non-rumour":>12}  {"Diff %":>8}')
print('-' * 58)
for key, label in [
    ('depths', 'Mean depth'),
    ('widths', 'Mean width'),
    ('ratios', 'Mean d/w ratio'),
    ('sizes',  'Mean size'),
]:
    rv  = r_arr[key].mean()
    nrv = nr_arr[key].mean()
    pct = 100 * (rv - nrv) / (nrv + 1e-9)
    print(f'{label:<22}  {rv:>10.3f}  {nrv:>12.3f}  {pct:>+8.1f}%')


Metric                      Rumour    Non-rumour    Diff %
----------------------------------------------------------
Mean depth                   3.914         6.724     -41.8%
Mean width                 566.647       494.695     +14.5%
Mean d/w ratio               0.022         0.090     -75.6%
Mean size                  721.201       876.169     -17.7%


In [18]:
# ── Mann-Whitney U tests: rumour vs non-rumour within Weibo ──────────────────
print('Mann-Whitney U: Rumour vs Non-rumour (within Weibo)')
print()
for key, label in [
    ('depths', 'Cascade depth     '),
    ('widths', 'Cascade width     '),
    ('ratios', 'Depth/width ratio '),
    ('sizes',  'Cascade size      '),
]:
    U, p = stats.mannwhitneyu(r_arr[key], nr_arr[key], alternative='two-sided')
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    d    = abs(r_arr[key].mean() - nr_arr[key].mean()) / \
           (((r_arr[key].std()**2 + nr_arr[key].std()**2) / 2)**0.5 + 1e-9)
    print(f'  {label}  U={U:7.0f}  p={p:.3e}  {sig:<3}  Cohen\'s d={d:.3f}')
print()
print('*** p<0.001  ** p<0.01  * p<0.05  ns=not significant')
# ── Mann-Whitney U tests: rumour vs non-rumour within Weibo ──────────────────
print('Mann-Whitney U: Rumour vs Non-rumour (within Weibo)')
print()
for key, label in [
    ('depths', 'Cascade depth     '),
    ('widths', 'Cascade width     '),
    ('ratios', 'Depth/width ratio '),
    ('sizes',  'Cascade size      '),
]:
    U, p = stats.mannwhitneyu(r_arr[key], nr_arr[key], alternative='two-sided')
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    d    = abs(r_arr[key].mean() - nr_arr[key].mean()) / \
           (((r_arr[key].std()**2 + nr_arr[key].std()**2) / 2)**0.5 + 1e-9)
    print(f'  {label}  U={U:7.0f}  p={p:.3e}  {sig:<3}  Cohen\'s d={d:.3f}')
print()
print('*** p<0.001  ** p<0.01  * p<0.05  ns=not significant')


Mann-Whitney U: Rumour vs Non-rumour (within Weibo)

  Cascade depth       U=1116141  p=1.549e-240  ***  Cohen's d=0.833
  Cascade width       U=3503324  p=9.016e-102  ***  Cohen's d=0.041
  Depth/width ratio   U= 985734  p=2.478e-281  ***  Cohen's d=0.816
  Cascade size        U=2999590  p=2.249e-23  ***  Cohen's d=0.059

*** p<0.001  ** p<0.01  * p<0.05  ns=not significant
Mann-Whitney U: Rumour vs Non-rumour (within Weibo)

  Cascade depth       U=1116141  p=1.549e-240  ***  Cohen's d=0.833
  Cascade width       U=3503324  p=9.016e-102  ***  Cohen's d=0.041
  Depth/width ratio   U= 985734  p=2.478e-281  ***  Cohen's d=0.816
  Cascade size        U=2999590  p=2.249e-23  ***  Cohen's d=0.059

*** p<0.001  ** p<0.01  * p<0.05  ns=not significant


In [19]:
# ── Side-by-side box plots ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#F8F9FA')

RUMOUR_COLOR     = '#FF595E'
NONRUMOUR_COLOR  = '#3A86FF'

for ax, (key, xlabel) in zip(axes, [
    ('depths', 'Cascade Depth'),
    ('widths', 'Cascade Width'),
    ('ratios', 'Depth / Width Ratio'),
]):
    bp = ax.boxplot(
        [r_arr[key], nr_arr[key]],
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markerfacecolor='#aaa',
                        markersize=3, linestyle='none', alpha=0.4),
    )
    bp['boxes'][0].set_facecolor(RUMOUR_COLOR)
    bp['boxes'][1].set_facecolor(NONRUMOUR_COLOR)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Rumour', 'Non-rumour'], fontsize=9)
    ax.set_ylabel(xlabel, fontsize=9)
    ax.set_facecolor('#FFFFFF')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Within-Weibo: Rumour vs Non-rumour Cascade Structure\n'
    '(boxes = IQR, line = median, whiskers = 1.5×IQR)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('weibo_label_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved weibo_label_comparison.png')


Saved weibo_label_comparison.png
